# E12 — generation on the GPU, from `final.pt` (nothing is scored here)

Registered in `research/native/PREREG_E12.md`, D2 and D7. It writes 150 files,
540,000 URLs, all with the baseline's raw decode (temperature 1.0, nothing else),
from each arm's **`final.pt`** (step 20,000; the owner's decision):
- **free:** 5 × 8,000 per class at the canonical seeds. The plan is supplied from the
  data, and the model writes the URL. For `e12_base` that is E11's plan (path and
  query); the slash arms also get the slash from the data;
- **control:** native, SC1's state tables, E8's seeds, the request supplied (the
  slash arms get S from the data).

The free files come first.

**Attach two inputs:**
1. **`uie-e12-code`**, the same dataset the training used;
2. the **E12_train** notebook's **output** (the three arm folders and
   `PROVENANCE.json`).

**Settings:** GPU T4, Internet **On**. **Save Version → Save & Run All.** About
5 h (E11's 200 files took 6.4 h). A rerun keeps every finished file with its
record. When done, download `/kaggle/working/e12_gen`.

In [ ]:
# --- 1. locate the inputs and verify the three checkpoints ------------------------
import hashlib, json
from pathlib import Path


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


ARMS = ("e12_base", "e12_final", "e12_no_r1")
root = Path("/kaggle/input")
code_hits = sorted(root.rglob("E12_CODE_MANIFEST.json")) if root.exists() else []
train_prov = sorted(p for p in root.rglob("PROVENANCE.json")
                    if all((p.parent / a).is_dir() for a in ARMS)) if root.exists() else []
if not code_hits or not train_prov:
    raise SystemExit("Attach both inputs (right panel -> Add Input):\n"
                     "  uie-e12-code          -- has E12_CODE_MANIFEST.json\n"
                     "  the E12_train OUTPUT  -- the three arm folders and PROVENANCE.json\n"
                     "Inputs mount when the session starts; rerun after attaching.")
CODE, TRAIN = code_hits[0].parent, train_prov[0].parent
tprov = json.loads(train_prov[0].read_text())
CKPTS = {}
for arm in ARMS:
    p = TRAIN / arm / "final.pt"
    assert p.is_file() and sha256(p) == tprov["arms"][arm]["final_sha256"], \
        f"{arm}: final.pt is not the one E12_train recorded"
    CKPTS[arm] = p
print("code:", CODE)
print("checkpoints verified (final.pt):", {k: str(v) for k, v in CKPTS.items()})

In [ ]:
# --- 2. environment: install first, then run THIS code and prove it ------------
import hashlib, importlib, json, subprocess, sys
from importlib import metadata

TLDEXTRACT = "5.3.1"          # the version the local evaluation parses with


def installed(dist):
    try:
        return metadata.version(dist)
    except metadata.PackageNotFoundError:
        return None


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


if installed("tldextract") != TLDEXTRACT:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        f"tldextract=={TLDEXTRACT}"], check=True)
    except subprocess.CalledProcessError:
        raise SystemExit("pip could not install tldextract: Settings -> Internet -> On, rerun.")
    importlib.invalidate_caches()
assert installed("tldextract") == TLDEXTRACT, installed("tldextract")

# CODE ONLY on the path: the data bundle carries an older `uie`.
sys.path.insert(0, str(CODE / "research" / "native"))
sys.path.insert(0, str(CODE))
for name in [m for m in sys.modules
             if m == "uie" or m.startswith("uie.") or m in ("e8_common", "e11_common", "e12_common")]:
    del sys.modules[name]
import uie
import e12_common as K
manifest = json.loads((CODE / "E12_CODE_MANIFEST.json").read_text())
for mod in (uie, K):
    assert str(CODE) in mod.__file__, f"{mod.__name__} is SHADOWED -- check sys.path"
bad = [f for f, h in manifest["files"].items() if sha256(CODE / f) != h]
bad += [t for t, h in manifest["tables"].items() if sha256(CODE / t) != h]
assert not bad, f"STOP. files differ from the manifest: {bad}"
print(f"code: commit {manifest['commit'][:12]}; {len(manifest['files'])} files and "
      f"{len(manifest['tables'])} tables verified")

import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4, then rerun.")
print("torch:", torch.__version__, "| gpu:", torch.cuda.get_device_name(0))

In [ ]:
# --- 3. tables, the registered jobs, the models ------------------------------------
from uie.controlled import StateSuffixTables
from uie.suffixes import SuffixTable
from uie.train import load_checkpoint

assert tuple(K.ARMS) == ARMS
OUT = Path("/kaggle/working/e12_gen")
OUT.mkdir(parents=True, exist_ok=True)
shipped = SuffixTable.load(CODE / "suffix_table.json")
tables = StateSuffixTables.load(CODE / "state_suffix_tables.json")
presence = json.loads((CODE / "presence_table.json").read_text())                 # e12_base
presence_slash = json.loads((CODE / "presence_slash_table.json").read_text())["table"]  # D2
device = torch.device("cuda")
ORDER = [(a, "free") for a in ARMS] + [(a, "control") for a in ARMS]
jobs = sorted(K.jobs(), key=lambda j: (ORDER.index((j["arm"], j["mode"])), j["label"],
                                       j["state"] or "", j["g"]))
print(f"{len(jobs)} files, {sum(j['n'] for j in jobs):,} URLs, free first")
_loaded = {}


def model_for(arm):
    if arm not in _loaded:
        _loaded.clear()
        torch.cuda.empty_cache()
        model, tok, _ = load_checkpoint(CKPTS[arm], device=device)
        model.to(device).eval()
        assert next(model.parameters()).device.type == "cuda"
        slash = arm in K.SLASH_ARMS
        assert tok.include_fields and tok.root_slash and tok.presence_plan
        assert bool(getattr(tok, "slash_plan", False)) == slash
        assert tok.vocab_size == (110 if slash else 108)
        _loaded[arm] = (model, tok)
        print(f"   loaded {arm}: vocab {tok.vocab_size}, max_len {model.cfg.max_len}", flush=True)
    return _loaded[arm]

In [ ]:
# --- 4. generate ----------------------------------------------------------------------
# PROVENANCE.json is rewritten after EVERY file; a rerun keeps a file only with a
# matching record -- anything else is made again: same seed, same bytes.
import platform
import time

PROV = OUT / "PROVENANCE.json"
prov = json.loads(PROV.read_text()) if PROV.exists() else {"files": {}}
prov.update({"purpose": "E12 (research/native/PREREG_E12.md); generation only",
             "checkpoint_file": "final.pt",
             "code_commit": manifest["commit"],
             "checkpoints": {a: sha256(p) for a, p in CKPTS.items()},
             "presence_table": sha256(CODE / "presence_table.json"),
             "presence_slash_table": sha256(CODE / "presence_slash_table.json"),
             "environment": {"torch": torch.__version__, "python": platform.python_version(),
                             "gpu": torch.cuda.get_device_name(0),
                             "tldextract": metadata.version("tldextract")},
             "decode": K.decode()})
t_start = time.time()
done = 0
for j in jobs:
    path = OUT / j["file"]
    rec = prov["files"].get(j["file"])
    if rec and path.is_file() and sha256(path) == rec["sha256"]:
        done += 1
        continue
    model, tok = model_for(j["arm"])
    t0 = time.time()
    trace = []
    if j["mode"] == "free":
        urls = K.free_urls(model, tok, j["arm"], j["label"], j["g"], shipped, presence=presence,
                           presence_slash=presence_slash, device=device, plan_trace=trace)
    else:
        urls = K.control_urls(model, tok, j["arm"], j["label"], j["state"], j["g"], tables,
                              presence_slash=presence_slash, device=device, plan_trace=trace)
    assert len(urls) == j["n"], (j["file"], len(urls))
    path.write_text(json.dumps(urls))
    prov["files"][j["file"]] = {**{k: j[k] for k in ("arm", "mode", "label", "state", "g",
                                                    "n", "seed", "plan")},
                                "sha256": sha256(path), "seconds": round(time.time() - t0, 1),
                                "plan_trace_mean": sum(trace) / len(trace) if trace else None}
    PROV.write_text(json.dumps(prov, indent=1))
    done += 1
    el = (time.time() - t_start) / 60
    print(f"  {done:>3}/{len(jobs)}  {j['file']:<42} {time.time() - t0:6.0f}s  "
          f"({el:.0f} min so far)", flush=True)
print(f"\nall {len(jobs)} files written. Now download /kaggle/working/e12_gen.")